# Load data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec
%matplotlib qt

### functions

In [ ]:
def compute_pos(step_size, image_size):
    # Generate positions for updating symmetry image
    positions = []
    for j in range(image_size):
        for i in range(0, image_size, step_size):
            positions.append(np.array([i, j]))
    return np.array(positions)

# def convert_ij_to_pq(i, j, kernel_size):
#     # Convert (i, j) to (p, q) with kernel size
#     p = i + (kernel_size - 1) / 2
#     q = j + (kernel_size - 1) / 2
#     return [int(p), int(q)]

def update_symmetry(animated_image, image, i, j, step_size, image_size):
    # Update part of the animated image based on the original image
    updated_i = np.min([i + step_size, image_size])
    animated_image[j][i:updated_i] = image[j][i:updated_i]
    return animated_image

def update_kernal(display_image, p, q, kernel_size, kernal_thickness):
    # Draw the sliding kernel onto the display image
    display_image[q:q + kernal_thickness, p:p + kernel_size] = 255
    display_image[q + kernel_size - kernal_thickness:q + kernel_size, p:p + kernel_size] = 255
    display_image[q:q + kernel_size, p:p + kernal_thickness] = 255
    display_image[q:q + kernel_size, p + kernel_size - kernal_thickness:p + kernel_size] = 255
    return display_image

def normalize_imgs_to_255(img):
    img-=img.min()
    img/=img.max()
    img = img*255
    return img

## load an image

In [ ]:
img = np.load('.\\data\\STEM img.npy')
r3 = np.load('.\\data\\rot3 img.npy')
r6 = np.load('.\\data\\rot6 img.npy')

### parameters setting

In [ ]:
image_size = r3.shape[0]
step_size = 120
kernel_size = 27
kernal_thickness = 3

In [ ]:
kernel_border = int((kernel_size-1) / 2)

In [ ]:
r3 = r3[kernel_border:image_size-kernel_border, kernel_border:image_size-kernel_border]
r6 = r6[kernel_border:image_size-kernel_border, kernel_border:image_size-kernel_border]
image_size = r3.shape[0]

In [ ]:
r3 = normalize_imgs_to_255(r3)
r6 = normalize_imgs_to_255(r6)
img = normalize_imgs_to_255(img)

In [ ]:
animated_image1 = np.full_like(r3, 255)
animated_image2 = np.full_like(r6, 255)
white_img = np.full_like(r3, 255)
display_image = img.copy()

# Create the figure and subplots
fig = plt.figure(figsize=(30, 6), dpi=80)
gs = GridSpec(1, 4)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[3])
ax3 = fig.add_subplot(gs[1])
ax4 = fig.add_subplot(gs[2])

# Adjust layout
ax1.axis("off")
ax2.axis("off")
ax3.axis("off")
ax4.axis("off")

pos1 = ax1.get_position()  
pos2 = ax3.get_position()  
pos3 = ax4.get_position()  

ax1.set_position([pos1.x0, pos1.y0, pos1.width, pos1.height])  
ax3.set_position([pos1.x0 + 2 * (pos2.x0 - pos1.x0), pos2.y0, pos2.width, pos2.height])
ax4.set_position([pos3.x0 + 1 * (pos2.x0 - pos1.x0), pos3.y0, pos3.width, pos3.height])


# Left image (sliding kernel visualization)
im1 = ax1.imshow(display_image,cmap = 'gray', vmin=0, vmax=255)
# Right image (gradually revealed pixels)
im_white = ax2.imshow(white_img, cmap = 'gray',vmin=0, vmax=255)
im2 = ax3.imshow(animated_image1, cmap = 'gray',vmin=0, vmax=255)
im3 = ax4.imshow(animated_image2, cmap = 'gray',vmin=0, vmax=255)

# Compute positions for the sliding kernel
positions = compute_pos(step_size, image_size)

def update(frame):
    global frame_idx, display_image, animated_image1,animated_image2
    i, j = positions[frame]
    display_image = img.copy()
    animated_image1 = update_symmetry(animated_image1, r3, i, j, step_size, image_size)
    animated_image2 = update_symmetry(animated_image2, r6, i, j, step_size, image_size)
    display_image = update_kernal(display_image, i, j, kernel_size, kernal_thickness)
    
    im1.set_array(display_image)
    im2.set_array(animated_image1)
    im3.set_array(animated_image2)
    return [im1, im2, im3]

# Create the animation
ani = FuncAnimation(
    fig,
    update,
    frames=len(positions) - 1,
    interval=1,  # Interval between frames in milliseconds
    blit=True,
)

# Save the animation as a GIF
ani.save(
    "rotational_symmetry_animation.gif",
    writer=PillowWriter(fps=800),
    savefig_kwargs={"transparent": True, "pad_inches": 0},
)

plt.close(fig)